# Advanced RAG: Structured Metadata Filtering & Self-Querying 🏷️
### Part of the `RAG-Journey` Portfolio Series

## The Problem
Standard vector search relies solely on semantic similarity. If a user asks for a specific document category or timeline (e.g., "Show me 2026 reports about NLP"), semantic search might pull highly similar documents from 2024 or 2025, completely ignoring the structural constraints.

## The Solution
1. **Metadata Filtering:** Hardcoding explicit structural constraints (e.g., `{"year": 2026}`) to immediately isolate a specific subset of data in your vector database before calculating semantic distance.
2. **Self-Querying Retrieval:** Utilizing an LLM to parse raw natural language, extract structural metadata criteria automatically, and format it into a dynamic filter query for the underlying vector store.

In [23]:
# 1. Install required packages
!pip install --quiet langchain-community langchain-core faiss-cpu langchain-groq lark langchain-huggingface databricks-langchain

# 2. Setup Kaggle Secrets for API Keys
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")
os.environ["HF_TOKEN"] = user_secrets.get_secret("HUGGING_FACE_HUB_TOKEN")
os.environ["HF_USERNAME"] = "MrJouH4"

# 3. Standard Imports (LCEL & Community Extensions)
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_groq import ChatGroq
from databricks_langchain import DatabricksVectorSearch

# Self-Querying Specific Imports
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo

## Step 1: Initialize Mock Corporate Technical Logs
We create an explicitly structured collection of technical project summaries spanning different years, tech stacks, and tracking categories.

In [11]:
docs = [
    Document(
        page_content="Deployed a bilingual NLP pipeline for real-time translation and sentiment analysis using Hugging Face.",
        metadata={"year": 2024, "topic": "NLP", "status": "production"}
    ),
    Document(
        page_content="Initiated research on Agentic RAG guardrails and dynamic prompt routing systems using LangChain.",
        metadata={"year": 2026, "topic": "RAG", "status": "research"}
    ),
    Document(
        page_content="Implemented context optimization and reranking notebooks to baseline pipeline metrics using RAGAS.",
        metadata={"year": 2026, "topic": "RAG", "status": "production"}
    ),
    Document(
        page_content="Configured automated computer vision systems for image restoration tasks using NAFNet modules.",
        metadata={"year": 2024, "topic": "CV", "status": "production"}
    )
]

# Build FAISS Vector Database
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 2: Explicit/Manual Metadata Filtering
Here, we interact directly with FAISS using native keyword arguments to force the retriever to only look at documents satisfying `{"year": 2026}`.

In [12]:
query = "Show me production-grade software developments"

# Define manual filter targeting specific metadata fields
manual_filter = {"year": 2026}

# Instantiate standard retriever passing search_kwargs
manual_retriever = vectorstore.as_retriever(
    search_kwargs={"filter": manual_filter, "k": 2}
)

results = manual_retriever.invoke(query)

print(f"--- MANUAL FILTER RESULTS (Year == 2026) ---")
for i, doc in enumerate(results):
    print(f"Match {i+1}: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")

--- MANUAL FILTER RESULTS (Year == 2026) ---
Match 1: Implemented context optimization and reranking notebooks to baseline pipeline metrics using RAGAS.
Metadata: {'year': 2026, 'topic': 'RAG', 'status': 'production'}

Match 2: Initiated research on Agentic RAG guardrails and dynamic prompt routing systems using LangChain.
Metadata: {'year': 2026, 'topic': 'RAG', 'status': 'research'}



## Step 3: Implementing a Dynamic Self-Querying Retriever
Instead of hardcoding the filters, we tell the LLM exactly what our metadata fields represent. The LLM will parse user queries like *"What are we researching in 2026?"* and dynamically build the corresponding metadata query internally.

In [22]:
# 1. Define metadata structural schemas for the LLM
metadata_field_info = [
    AttributeInfo(
        name="year",
        description="The calendar year the project or document was generated",
        type="integer",
    ),
    AttributeInfo(
        name="topic",
        description="The specific domain of the technical task, choose from ['NLP', 'RAG', 'CV']",
        type="string",
    ),
    AttributeInfo(
        name="status",
        description="The operational lifecycle state of the model, choose from ['production', 'research']",
        type="string",
    ),
]

document_content_description = "Technical summaries of corporate artificial intelligence initiatives"

# 2. Instantiate LLM with a stable, zero temperature
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# 3. Create the SelfQueryRetriever
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    # verbose=True # Allows us to inspect the generated internal query format
)

ImportError: cannot import name 'DatabricksVectorSearch' from 'langchain_community.vectorstores' (/usr/local/lib/python3.12/dist-packages/langchain_community/vectorstores/__init__.py)

## Step 3: Bypassing High-Level Wrappers with a Custom LCEL Self-Query Chain
Because high-level wrappers like `SelfQueryRetriever` contain rigid, monolithic dependencies that easily break across version updates, we will implement a custom, transparent **Dynamic Query Constructor** using pure LangChain Expression Language (LCEL) and Pydantic.

This gives us complete control over how natural language is translated into semantic search parameters and metadata filters.

In [24]:
from pydantic import BaseModel, Field
from typing import Optional, Dict, Any
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. Define the desired structured output schema using Pydantic
class QuerySchema(BaseModel):
    query: str = Field(description="The core semantic search text focused on the topic or concept, stripped of filter keywords like dates or status.")
    year: Optional[int] = Field(None, description="The specific calendar year requested (e.g. 2024 or 2026), if mentioned.")
    status: Optional[str] = Field(None, description="The operational state filter. Must be exactly 'production' or 'research' if implied.")

# 2. Build a highly specific system prompt for structured extraction
parser = JsonOutputParser(pydantic_object=QuerySchema)

prompt_template = """You are a precise query translation engine. Your job is to convert a user's natural language request into a structured JSON query object for a vector database.

Available Metadata Fields:
- year: Integer (e.g., 2024, 2026)
- status: String ('production' or 'research')

Format Instructions:
{format_instructions}

User Request: {user_request}
Structured JSON Output:"""

prompt = ChatPromptTemplate.from_template(
    template=prompt_template,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 3. Instantiate LLM and chain
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# The query construction chain
query_generation_chain = prompt | llm | parser

## Step 4: Building the Custom Retrieval Logic & Execution
Now, we create an explicit function that invokes our constructor chain, extracts the metadata arguments, maps them to native FAISS syntax, and executes the search.

In [26]:
def custom_self_query_search(user_query: str, vector_store, k=2):
    # Execute the LCEL chain to get the structured arguments
    structured_args = query_generation_chain.invoke({"user_request": user_query})
    
    print("--- INTERNAL LLM PARSING ANALYSIS ---")
    print(f"Extracted Structured Schema: {structured_args}\n")
    
    # Isolate semantic query and assemble filters
    semantic_query = structured_args.get("query", user_query)
    
    # Construct native FAISS filter dictionary
    native_filter = {}
    if structured_args.get("year"):
        native_filter["year"] = structured_args["year"]
    if structured_args.get("status"):
        native_filter["status"] = structured_args["status"]
        
    # Execute search with conditional filtering
    if native_filter:
        print(f"Applying Dynamic Filter to FAISS: {native_filter}")
        retriever = vector_store.as_retriever(search_kwargs={"filter": native_filter, "k": k})
    else:
        print("No structural metadata filters detected. Executing raw semantic search.")
        retriever = vector_store.as_retriever(search_kwargs={"k": k})
        
    return retriever.invoke(semantic_query)


# --- TEST CASE 1: Expects topic filtering and status filtering ---
test_query_1 = "Find production updates regarding RAG pipelines"
results_1 = custom_self_query_search(test_query_1, vectorstore)

print("\n--- TARGETED SEARCH RESULTS ---")
for i, doc in enumerate(results_1):
    print(f"Match {i+1}: {doc.page_content} | Metadata: {doc.metadata}")

--- INTERNAL LLM PARSING ANALYSIS ---
Extracted Structured Schema: {'query': 'RAG pipelines updates', 'year': None, 'status': 'production'}

Applying Dynamic Filter to FAISS: {'status': 'production'}

--- TARGETED SEARCH RESULTS ---
Match 1: Implemented context optimization and reranking notebooks to baseline pipeline metrics using RAGAS. | Metadata: {'year': 2026, 'topic': 'RAG', 'status': 'production'}
Match 2: Deployed a bilingual NLP pipeline for real-time translation and sentiment analysis using Hugging Face. | Metadata: {'year': 2024, 'topic': 'NLP', 'status': 'production'}
